# Make sure data is not biased

In [23]:
import pandas as pd
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

from helpers.logger import log_step, save_log
from helpers.data_loader import DataLoader
from helpers.profile import profile

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 180)

In [24]:
GEO_BLOCKED_CSV_PATH = DataLoader.interim("geo_blocked.csv")

df_raw = pd.read_csv(GEO_BLOCKED_CSV_PATH)
df_clean = df_raw.copy()
profile(df_clean, "Starting point right after removing geo blocks")

log_step(
    step='Initial Load',
    rows_before=df_raw.shape[0],
    rows_after=df_clean.shape[0],
    cols_before=df_raw.shape[1],
    cols_after=df_clean.shape[1]
)

=== Starting point right after removing geo blocks ===
shape: (196676, 63)
full duplicates: 0
duplicate Inspection ID: 0


First load done, now we expore bias in data, since it is sorted by the `_inspection_date`

In [25]:
# Unnamed: 0,Inspection ID,DBA Name,AKA Name,License #,Facility Type,Risk,Address,Zip,Inspection Date,Inspection Type,Results,Violations,Latitude,Longitude,Historical Wards 2003-2015,Zip Codes,Community Areas,Census Tracts,Wards,BL_ID,BL_LICENSE_ID,ACCOUNT NUMBER,SITE NUMBER,BL_LEGAL_NAME,BL_DBA_NAME,BL_ADDRESS,BL_CITY,BL_STATE,BL_ZIP_CODE,WARD,PRECINCT,WARD PRECINCT,POLICE DISTRICT,COMMUNITY AREA,COMMUNITY AREA NAME,NEIGHBORHOOD,LICENSE CODE,LICENSE DESCRIPTION,BUSINESS ACTIVITY ID,BUSINESS ACTIVITY,LICENSE NUMBER,APPLICATION TYPE,APPLICATION CREATED DATE,APPLICATION REQUIREMENTS COMPLETE,PAYMENT DATE,CONDITIONAL APPROVAL,LICENSE TERM START DATE,LICENSE TERM EXPIRATION DATE,LICENSE APPROVED FOR ISSUANCE,DATE ISSUED,LICENSE STATUS,LICENSE STATUS CHANGE DATE,SSA,BL_LATITUDE,BL_LONGITUDE,BL_LOCATION,flag_non_il_state,flag_non_chicago_city,flag_longitude_outside_typical_range,violations_recorded,license_matched,has_prior_inspection

df_clean['Inspection Date'] = pd.to_datetime(df_clean['Inspection Date'], errors='coerce')
df_clean['inspection_year'] = df_clean['Inspection Date'].dt.year

valid_results = ['Pass', 'Fail', 'Pass w/ Conditions']
df_results = df_clean[df_clean['Results'].isin(valid_results)].copy()

yearly_counts = (
    df_results
    .groupby(['inspection_year', 'Results'])
    .size()
    .unstack(fill_value=0) # mel a5er by7ot el column ely 3al shemal 5ales mara wa7da
    .sort_index()
)
print("=== Raw Counts by Year ===")
print(yearly_counts)

yearly_props = yearly_counts.div(yearly_counts.sum(axis=1), axis=0).round(4) * 100
print("\n=== Class Distribution (%) by Year ===")
print(yearly_props)


=== Raw Counts by Year ===
Results          Fail   Pass  Pass w/ Conditions
inspection_year                                 
2010             4504  11453                1425
2011             4355  11713                1516
2012             3642  10845                1531
2013             3349  11964                1755
2014             3719  12593                2309
2015             3602  12108                2155
2016             4294  12910                2634
2017             4119  12226                2135
2018             3312   6356                5137
2019             3159   3835                6815

=== Class Distribution (%) by Year ===
Results           Fail   Pass  Pass w/ Conditions
inspection_year                                  
2010             25.91  65.89                8.20
2011             24.77  66.61                8.62
2012             22.74  67.71                9.56
2013             19.62  70.10               10.28
2014             19.97  67.63               1

### Calculate pass rate by year

In [26]:
if 'Pass' in yearly_counts.columns:
    yearly_counts['pass_total'] = yearly_counts.get('Pass', 0) + yearly_counts.get('Pass w/ Conditions', 0)

    pass_rate = (yearly_counts['pass_total'] / yearly_counts.iloc[:, :-2].sum(axis=1)).round(4)
    print("\n=== Pass Rate by Year ===")
    print(pass_rate.to_frame("pass_rate_%").assign(**{"pass_rate_%": lambda x: x * 100}))


=== Pass Rate by Year ===
                 pass_rate_%
inspection_year             
2010                   80.70
2011                   82.33
2012                   85.43
2013                   89.59
2014                   91.36
2015                   90.79
2016                   90.35
2017                   87.86
2018                  118.88
2019                  152.27


In [27]:
from scipy.stats import chi2_contingency

chi2, p, dof, expected = chi2_contingency(yearly_counts[valid_results].dropna(axis=1))
print(f"\n=== Chi-Square Test (year vs result distribution) ===")
print(f"Chi2={chi2:.2f}, p-value={p:.4f}, dof={dof}")
print(" Significant temporal bias detected!" if p < 0.05 else "No significant temporal bias detected.")


cutoff_year = df_results['inspection_year'].max() - 1   # last 2 years = test
df_train = df_results[df_results['inspection_year'] <  cutoff_year]
df_test  = df_results[df_results['inspection_year'] >= cutoff_year]

print(f"\n=== Proposed Split (cutoff year ≥ {cutoff_year}) ===")
print(f"Train: {len(df_train):,} rows  |  {df_train['inspection_year'].min()}–{cutoff_year - 1}")
print(f"Test:  {len(df_test):,} rows   |  {cutoff_year}–{df_results['inspection_year'].max()}")

print("\nTrain class distribution (%):")
print((df_train['Results'].value_counts(normalize=True) * 100).round(2))
print("\nTest class distribution (%):")
print((df_test['Results'].value_counts(normalize=True) * 100).round(2))

profile(df_clean, "After temporal bias check")
log_step(
    step='Temporal Bias Check',
    rows_before=df_clean.shape[0],
    rows_after=df_clean.shape[0],
    cols_before=df_clean.shape[1],
    cols_after=df_clean.shape[1] + 1  # +inspection_year
)


=== Chi-Square Test (year vs result distribution) ===
Chi2=19939.63, p-value=0.0000, dof=18
 Significant temporal bias detected!

=== Proposed Split (cutoff year ≥ 2018) ===
Train: 142,856 rows  |  2010–2017
Test:  28,614 rows   |  2018–2019

Train class distribution (%):
Results
Pass                  67.07
Fail                  22.11
Pass w/ Conditions    10.82
Name: proportion, dtype: float64

Test class distribution (%):
Results
Pass w/ Conditions    41.77
Pass                  35.62
Fail                  22.61
Name: proportion, dtype: float64
=== After temporal bias check ===
shape: (196676, 64)
full duplicates: 0
duplicate Inspection ID: 0


Converting Pass w/ Conditions to PASS

In [ ]:
target_map = {
    'Pass':               'PASS',
    'Pass w/ Conditions': 'PASS',
    'Fail':               'FAIL',
}
df_results['target'] = df_results['Results'].map(target_map)

print("=== Binary Target Distribution by Year ===")
binary_by_year = (
    df_results.groupby(['inspection_year', 'target'])
    .size()
    .unstack(fill_value=0)
    .assign(pass_rate=lambda x: (x['PASS'] / x.sum(axis=1) * 100).round(2))
)
print(binary_by_year)

=== Binary Target Distribution by Year ===
target           FAIL   PASS  pass_rate
inspection_year                        
2010             4504  12878      74.09
2011             4355  13229      75.23
2012             3642  12376      77.26
2013             3349  13719      80.38
2014             3719  14902      80.03
2015             3602  14263      79.84
2016             4294  15544      78.35
2017             4119  14361      77.71
2018             3312  11493      77.63
2019             3159  10650      77.12


In [ ]:
from scipy.stats import chi2_contingency

binary_counts = (
    df_results.groupby(['inspection_year', 'target'])
    .size()
    .unstack(fill_value=0)
)

chi2_b, p_b, dof_b, _ = chi2_contingency(binary_counts)
print(f"Binary Chi2={chi2_b:.2f}, p={p_b:.4f}, dof={dof_b}")

print("\n=== Binary Pass Rate by Year ===")
print(binary_counts.assign(
    pass_rate=(binary_counts['PASS'] / binary_counts.sum(axis=1) * 100).round(2)
)[['PASS', 'FAIL', 'pass_rate']])

Binary Chi2=378.18, p=0.0000, dof=9

=== Binary Pass Rate by Year ===
target            PASS  FAIL  pass_rate
inspection_year                        
2010             12878  4504      74.09
2011             13229  4355      75.23
2012             12376  3642      77.26
2013             13719  3349      80.38
2014             14902  3719      80.03
2015             14263  3602      79.84
2016             15544  4294      78.35
2017             14361  4119      77.71
2018             11493  3312      77.63
2019             10650  3159      77.12


In [30]:
import numpy as np

n = binary_counts.sum().sum()
cramers_v = np.sqrt(chi2_b / (n * (min(binary_counts.shape) - 1)))
print(f"Cramér's V = {cramers_v:.4f}")  # < 0.1 = negligible, < 0.3 = weak

Cramér's V = 0.0470
